In [ ]:
from pathlib import Path
import sys

import torch
from torch.utils.data import DataLoader

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
SRC = ROOT / "src"
if str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))

from dataset import PatchDataset
from inference import GenerationOptions, MicroLadPredictor
from models import PatchVAE, DDPM, TimeUNet

DATA_DIR = ROOT / "data" / "images"
VAE_CKPT = ROOT / "microlad-anode" / "vae_anode.pth"
UNET_CKPT = ROOT / "microlad-anode" / "unet_anode.pth"
PATCH_SIZE = 64
AXIS = "z"
AXIS_ID = 0
SLICE_INDEX = 12
TIMESTEPS = 3
SDS_STEPS = 1
SEED = 0

In [ ]:
torch.manual_seed(SEED)
dataset = PatchDataset(DATA_DIR, patch_size=PATCH_SIZE, seed=SEED)
batch = next(iter(DataLoader(dataset, batch_size=1, shuffle=False)))
condition = batch
condition.shape

In [ ]:
vae = PatchVAE(latent_ch=4)
vae.load_state_dict(torch.load(VAE_CKPT, map_location="cpu")["vae"])
vae.eval()

unet = TimeUNet(latent_ch=4, base_ch=128, time_dim=64)
unet.load_state_dict(torch.load(UNET_CKPT, map_location="cpu"))
unet.eval()

ddpm = DDPM(timesteps=TIMESTEPS)

In [ ]:
options = GenerationOptions(
    volume_shape=(4, 16, 16, 16),
    sds_steps=SDS_STEPS,
    sds_unet=unet,
    t_min=1,
    t_max=TIMESTEPS,
    lock_condition_slice=True,
)
predictor = MicroLadPredictor(vae=vae, unet=unet, ddpm=ddpm, options=options)
result = predictor.predict(
    {
        "images": [{"image": condition, "axis": AXIS_ID, "index": SLICE_INDEX}],
        "condition_weight": 1.0,
        "stats_weight": 1.0,
    }
)

locked_delta = (result["volume"][SLICE_INDEX] - condition.squeeze(0)).abs().max()
assert result["volume"].shape == torch.Size([64, 1, 64, 64])
assert len(result["sds_history"]) == SDS_STEPS
result["volume"].shape, len(result["sds_history"]), float(locked_delta)